In [1]:
import originpro as op
import numpy as np
import pandas as pd

import os
# Very useful, especially during development, when you are
# liable to have a few uncaught exceptions.
# Ensures that the Origin instance gets shut down properly.
# Note: only applicable to external Python.
import sys
def origin_shutdown_exception_hook(exctype, value, traceback):
    '''Ensures Origin gets shut down if an uncaught exception'''
    op.exit()
    sys.__excepthook__(exctype, value, traceback)
if op and op.oext:
    sys.excepthook = origin_shutdown_exception_hook


# Set Origin instance visibility.
# Important for only external Python.
# Should not be used with embedded Python.
if op.oext:
    op.set_show(True)


# Example of opening a project and reading data.

# We'll open the Tutorial Data.opju project that ships with Origin.
src_opju = r"C:\usrspace\mywork\edges2.opju"   # <— 用绝对路径；注意路径和文件名是否正确
print("File exists? ", os.path.exists(src_opju))
ok = op.open(file=src_opju)
print("Project opened? ", ok)


File exists?  True
Project opened?  True
#Workbooks: 1 | #MatrixBooks: 0 | #Graphs: 38 | #Notes: 0
[Book1] -> ['pending_edges_ttb30', 'pending_edges_ttb40', 'pending_edges_ttb50', 'pending_edges_ttb60', 'pending_edges_ttb70', 'pending_edges_ttb80', 'pending_edges_ttb90', 'pending_edges_ttb100', 'pending_edges_ttb110', 'pending_edges_ttb120', 'pending_edges_ttb130', 'pending_edges_ttb140', 'stable_duration_time']
Active to: Book1
Sheet: pending_edges_ttb30
Sheet: pending_edges_ttb40
Sheet: pending_edges_ttb50
Sheet: pending_edges_ttb60
Sheet: pending_edges_ttb70
Sheet: pending_edges_ttb80
Sheet: pending_edges_ttb90
Sheet: pending_edges_ttb100
Sheet: pending_edges_ttb110
Sheet: pending_edges_ttb120
Sheet: pending_edges_ttb130
Sheet: pending_edges_ttb140
Sheet: stable_duration_time


In [2]:

# ===== 2) 列出所有页面类型，看看有没有工作簿 =====
wbooks = list(op.pages('w'))   # 所有工作簿（Worksheet Book）
mbks   = list(op.pages('m'))   # 所有矩阵簿（Matrix Book）
graphs = list(op.pages('g'))   # 所有图页（Graph），仅排查
notes  = list(op.pages('n'))   # 所有Notes，仅排查

print("#Workbooks:", len(wbooks), "| #MatrixBooks:", len(mbks), "| #Graphs:", len(graphs), "| #Notes:", len(notes))

# ===== 3) 如果有工作簿：打印每个工作簿下所有工作表名 =====
for wb in wbooks:
    sheet_names = [wks.name for wks in wb]
    print(f"[{wb.name}] -> {sheet_names}")

# ===== 4) 如果是矩阵簿：打印矩阵表名（有些项目只有矩阵簿，没有工作簿）=====
for mb in mbks:
    ms_names = [ms.name for ms in mb]
    print(f"[{mb.name}] (Matrix) -> {ms_names}")

# ===== 5) 若你想强行拿到第一个工作簿并遍历 =====
if wbooks:
    wb = wbooks[0]
    print("Active to:", wb.name)
    wb.activate()  # 让它成为激活窗口
    for wks in wb:
        print("Sheet:", wks.name)

#Workbooks: 1 | #MatrixBooks: 0 | #Graphs: 44 | #Notes: 0
[Book1] -> ['pending_edges_ttb30', 'pending_edges_ttb40', 'pending_edges_ttb50', 'pending_edges_ttb60', 'pending_edges_ttb70', 'pending_edges_ttb80', 'pending_edges_ttb90', 'pending_edges_ttb100', 'pending_edges_ttb110', 'pending_edges_ttb120', 'pending_edges_ttb130', 'pending_edges_ttb140', 'stable_duration_time', 'pending_edges_ttb10', 'pending_edges_ttb20']
Active to: Book1
Sheet: pending_edges_ttb30
Sheet: pending_edges_ttb40
Sheet: pending_edges_ttb50
Sheet: pending_edges_ttb60
Sheet: pending_edges_ttb70
Sheet: pending_edges_ttb80
Sheet: pending_edges_ttb90
Sheet: pending_edges_ttb100
Sheet: pending_edges_ttb110
Sheet: pending_edges_ttb120
Sheet: pending_edges_ttb130
Sheet: pending_edges_ttb140
Sheet: stable_duration_time
Sheet: pending_edges_ttb10
Sheet: pending_edges_ttb20


下面是写入测试

In [ ]:
# 写入测试
# 1) 找到或创建 Book1
wb = op.find_book('w', 'Book1')
if wb is None:
    wb = op.new_book('w', lname='Book1')   # 没有就新建

# 2) 在 Book1 里新建空表，并激活到前台
try:
    wks = wb.add_sheet('MyEmptySheet', active=True)   # 某些版本支持 name + active
except TypeError:
    wks = wb.add_sheet()              # 兼容：如果不支持命名/active参数
    wks.name = 'MyEmptySheet'
    wks.activate()

wb.activate()                          # 激活工作簿窗口
print("Created:", wks.lt_range())

# 3)（可选）列出 Book1 里的所有表，确认新表存在
print("Book1 sheets:", [s.name for s in wb])


In [8]:
wks = op.find_sheet('w', '[Book1]pending_edges_ttb100')

In [9]:
wks.activate()  # 可选：激活这张表

1

In [12]:
print("形状 (rows, cols):", wks.shape)               # 来自 DSheet 的 shape 属性


形状 (rows, cols): (21894, 2)


In [13]:
df = wks.to_df(head='L')     # 也可以 head='C' 用注释行命名，或 head='' 用短名 A,B,C...
print(df.head())
print(df.dtypes)

   time  pending_edges
0   0.0           47.0
1   1.0           47.0
2   2.0           47.0
3   3.0           47.0
4   4.0           47.0
time             float64
pending_edges    float64
dtype: object


In [18]:
mapping = df.set_index('time')['pending_edges'].to_dict()

In [37]:
import draw.pymatlab2.origin_function.first as first
# 首先，我们检查这个时间是否是连续的
_,_,_,gaps =first.analyze_time_continuity(mapping)
# 其次，我们再把这个坐标转化为xy形状的


edges, _,_   =first.build_edges_from_mapping(mapping)

In [44]:
import draw.pymatlab2.origin_function.longest_zero_window_from_edges as longest_zero_window_from_edges
start_t, end_t, dur_s, dur_min = longest_zero_window_from_edges.longest_zero_window_from_edges(edges,0)
print(f"max durtion is{ dur_s} and its ")

win_list = longest_zero_window_from_edges.list_zero_windows(edges, 0)
avg = longest_zero_window_from_edges.average_zero_window_duration(edges, 0)
if avg is None:
    print("没有 zero-edge 窗口。")
else:
    avg_s, avg_min, cnt = avg
    print(f"平均窗口时长: {avg_s:.2f} 秒 (~{avg_min:.2f} 分钟)，基于 {cnt} 个窗口")

max durtion is88 and its 
平均窗口时长: 64.58 秒 (~1.08 分钟)，基于 120 个窗口


接下里我们遍历所有表，

In [3]:

import pandas as pd
from pathlib import Path

# 可调参数（随便改）
MIN_WINDOW_LEN = 1    # 只统计长度 >= 这个阈值的 0 窗口；若不想过滤就设为 1

results = []          # 用于最后汇总；你也可以直接 print 每一步

for wb in op.pages('w'):
    for wks in wb:
        nm = wks.name
        if not nm.startswith('pending_edges_'):
            continue

        # ===== 读取 =====
        df = wks.to_df(head='L')
        if not {'time','pending_edges'}.issubset(df.columns):
            print(f"跳过（缺列）: [{wb.name}] {nm}")
            continue

        # ===== 规范化 time/edge 到 int，并丢弃无法转化的 time =====
        # （若是浮点秒，你也可以改成 round 后再转 int）
        df = df[['time','pending_edges']].copy()
        try:
            df['time'] = df['time'].astype(int)
        except Exception:
            df['time'] = pd.to_numeric(df['time'], errors='coerce').dropna().astype(int)
        df['pending_edges'] = pd.to_numeric(df['pending_edges'], errors='coerce')

        # ===== 构建 mapping 并做连续性检查（只报告缺口，不在这一步补）=====
        mapping = df.set_index('time')['pending_edges'].to_dict()
        times = sorted(mapping.keys())
        if not times:
            print(f"跳过（空表）: [{wb.name}] {nm}")
            continue

        t_min, t_max = times[0], times[-1]
        gaps = []
        for a, b in zip(times, times[1:]):
            if b - a > 1:
                gaps.append((a+1, b-1))
        missing_total = sum((b - a + 1) for (a, b) in gaps) if gaps else 0

        # ===== 补缺口：把缺的秒都当作 0 =====
        size = t_max - t_min + 1
        edges = [0] * size    # 直接补 0（满足“断了要补上去”）
        for t, e in mapping.items():
            idx = t - t_min
            if pd.notna(e):
                # e 可能是浮点或字符串，尽量转 int
                try:
                    edges[idx] = int(e)
                except Exception:
                    try:
                        edges[idx] = int(float(e))
                    except Exception:
                        edges[idx] = 0

        # ===== 扫描获取所有 edge==0 的连续窗口（可用 MIN_WINDOW_LEN 过滤）=====
        windows = []  # [(start_t, end_t, length_s), ...]
        cur_len = 0
        cur_start_idx = None

        for i, v in enumerate(edges):
            if v == 0:
                if cur_len == 0:
                    cur_start_idx = i
                cur_len += 1
            else:
                if cur_len >= MIN_WINDOW_LEN and cur_start_idx is not None:
                    s = t_min + cur_start_idx
                    L = cur_len
                    e = s + L - 1
                    windows.append((s, e, L))
                cur_len = 0
                cur_start_idx = None

        # 收尾
        if cur_len >= MIN_WINDOW_LEN and cur_start_idx is not None:
            s = t_min + cur_start_idx
            L = cur_len
            e = s + L - 1
            windows.append((s, e, L))

        # ===== 最长窗口 =====
        if windows:
            s_long, e_long, L_long = max(windows, key=lambda x: x[2])
            longest_len_min = L_long / 60.0
        else:
            s_long = e_long = None
            L_long = 0
            longest_len_min = 0.0

        # ===== 平均窗口时长 =====
        if windows:
            avg_s = sum(L for _,_,L in windows) / len(windows)
            avg_min = avg_s / 60.0
            n_windows = len(windows)
        else:
            avg_s = avg_min = 0.0
            n_windows = 0

        # ===== 打印或汇总 =====
        print(f"[{wb.name}] {nm} | time[{t_min},{t_max}] "
              f"| gaps={len(gaps)} miss={missing_total}s "
              f"| longest={L_long}s ({longest_len_min:.2f}m) "
              f"| avg={avg_s:.2f}s ({avg_min:.2f}m) over {n_windows} windows")

        results.append({
            'book': wb.name, 'sheet': nm,
            'time_min': t_min, 'time_max': t_max,
            'observed_seconds': len(times),
            'theoretical_seconds': (t_max - t_min + 1),
            'n_gaps': len(gaps), 'missing_total_seconds': missing_total,
            'longest_start': s_long, 'longest_end': e_long,
            'longest_len_s': L_long, 'longest_len_min': round(longest_len_min, 4),
            'avg_len_s': round(avg_s, 4), 'avg_len_min': round(avg_min, 4),
            'n_windows': n_windows,
            'min_window_len_used': MIN_WINDOW_LEN,
        })

# ===== 可选：汇总保存 =====
if results:
    df_sum = pd.DataFrame(results)
    # 从名字中提取 ttb 数，方便排序
    def _pick_ttb(s):
        try:
            return int(''.join(ch for ch in s if ch.isdigit()))
        except Exception:
            return None
    df_sum['ttb'] = df_sum['sheet'].map(_pick_ttb)
    df_sum = df_sum.sort_values(['book','ttb'], na_position='last')

    cols = ['book','sheet','ttb',
            'time_min','time_max','observed_seconds','theoretical_seconds',
            'n_gaps','missing_total_seconds',
            'longest_start','longest_end','longest_len_s','longest_len_min',
            'avg_len_s','avg_len_min','n_windows','min_window_len_used']
    print("\n=== SUMMARY ===")
    print(df_sum[cols].to_string(index=False))

    out_csv = Path.cwd() / 'pending_edges_windows_summary.csv'
    df_sum[cols].to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"Saved: {out_csv}")
else:
    print("没有匹配到 pending_edges_* 的工作表。")


[Book1] pending_edges_ttb30 | time[3,21893] | gaps=0 miss=0s | longest=158s (2.63m) | avg=112.27s (1.87m) over 139 windows
[Book1] pending_edges_ttb40 | time[0,21893] | gaps=0 miss=0s | longest=148s (2.47m) | avg=114.16s (1.90m) over 129 windows
[Book1] pending_edges_ttb50 | time[0,21893] | gaps=0 miss=0s | longest=138s (2.30m) | avg=101.86s (1.70m) over 131 windows
[Book1] pending_edges_ttb60 | time[0,21893] | gaps=0 miss=0s | longest=128s (2.13m) | avg=92.98s (1.55m) over 128 windows
[Book1] pending_edges_ttb70 | time[0,21893] | gaps=0 miss=0s | longest=118s (1.97m) | avg=84.91s (1.42m) over 125 windows
[Book1] pending_edges_ttb80 | time[0,21893] | gaps=0 miss=0s | longest=108s (1.80m) | avg=79.06s (1.32m) over 125 windows
[Book1] pending_edges_ttb90 | time[0,21893] | gaps=0 miss=0s | longest=98s (1.63m) | avg=70.80s (1.18m) over 120 windows
[Book1] pending_edges_ttb100 | time[0,21893] | gaps=0 miss=0s | longest=88s (1.47m) | avg=64.58s (1.08m) over 120 windows
[Book1] pending_edges_

In [4]:
import originpro as op
import pandas as pd

# ——— 假定你已有 df_sum（含 ttb / longest_len_s / avg_len_s）———
df_plot = df_sum[['ttb','longest_len_s','avg_len_s']].copy()
df_plot = df_plot.sort_values('ttb')
df_plot.rename(columns={'ttb':'TTB',
                        'longest_len_s':'Longest_s',
                        'avg_len_s':'Average_s'}, inplace=True)

# 1) 打开/创建 Book1（注意 find_book 要写类型参数 'w'）
wb = op.find_book('w', 'Book1')
if wb is None:
    wb = op.new_book('w', lname='Book1')

# 2) 新建一个表（如果已经有同名，就直接覆盖写入）
target_name = 'pending_edges_summary'
target = None
for s in wb:
    if s.name == target_name or getattr(s, 'lname', '') == target_name:
        target = s
        break
if target is None:
    target = wb.add_sheet()
    target.name = target_name

# 3) 写入数据并激活
target.from_df(df_plot)   # 会带列头 TTB / Longest_s / Average_s
target.activate()
wb.activate()
print("写入完成：", target.lt_range())
print(df_plot.head())


写入完成： [Book1]pending_edges_summary
    TTB  Longest_s  Average_s
12   10        178   148.6194
13   20        168   136.0150
0    30        158   112.2662
1    40        148   114.1550
2    50        138   101.8626


上述过程就完成了数据表生产，后续绘图，就直接在origin里生成即可